In [1]:
import json
import os
import operator
import re
from pathlib import Path
from typing import Annotated, Any, Dict, List, TypedDict

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field


In [2]:
load_dotenv(dotenv_path=Path.cwd() / ".env")
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY"),
)


In [3]:
class PitchBrief(BaseModel):
    problem_statement: str
    target_persona: str
    value_proposition: str
    key_assumptions: List[str]


class VCMemo(BaseModel):
    invest_decision: str
    investment_thesis: str
    dealbreaker_risks: List[str]
    mandatory_milestones: List[str]
    final_partner_note: str


class StartupPitchState(TypedDict):
    raw_idea: str
    target_market: str
    funding_stage: str

    refined_pitch: PitchBrief
    competitor_analysis: str
    monetization_plan: str
    tech_blueprint: str
    regulatory_report: str

    domain_scores: Annotated[List[Dict[str, Any]], operator.add]
    red_flags: Annotated[List[str], operator.add]
    logs: Annotated[List[str], operator.add]
    pipeline_logs: Annotated[List[str], operator.add]

    final_average_score: float
    investment_verdict: str
    vc_memo: str


In [4]:
def parse_json_block(raw_text: str, model_cls: type[BaseModel]) -> BaseModel:
    text = (raw_text or "").strip()
    candidate = text

    if text.startswith("```"):
        match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.S | re.I)
        if match:
            candidate = match.group(1)
    elif not text.startswith("{") and not text.startswith("["):
        match = re.search(r"(\{.*\})", text, re.S)
        if match:
            candidate = match.group(1)

    data = json.loads(candidate)
    return model_cls.model_validate(data)


class DepartmentScore(BaseModel):
    department: str = Field(description="Department name")
    score: int = Field(ge=1, le=10, description="Score from 1 to 10")
    red_flag: str = Field(description="Single biggest risk or dealbreaker in this domain")


def extract_score_and_flag(report: str, department: str) -> Dict[str, Any]:
    score_match = re.search(r"(?im)^SCORE:\s*(\d+)\s*$", report)
    flag_match = re.search(r"(?im)^RED_FLAG:\s*(.+)$", report)
    return {
        "department": department,
        "score": int(score_match.group(1)) if score_match else 5,
        "red_flag": flag_match.group(1).strip() if flag_match else "No major red flag identified.",
    }


In [5]:
def idea_refiner(state: StartupPitchState) -> dict:
    messages = [
        SystemMessage(
            content=(
                "You are a Startup Incubator Partner. Convert the raw startup idea into a clear, structured Pitch Brief covering:\n"
                "1. Core Problem Statement\n"
                "2. Target Customer Persona\n"
                "3. Core Value Proposition\n"
                "4. Key Assumptions to Validate. Keep it crisp.\n"
                "Return only valid JSON using this exact schema:\n"
                "{\n"
                "  \"problem_statement\": \"str\",\n"
                "  \"target_persona\": \"str\",\n"
                "  \"value_proposition\": \"str\",\n"
                "  \"key_assumptions\": [\"str\"]\n"
                "}\n"
            )
        ),
        HumanMessage(
            content=(
                f"IDEA:{state['raw_idea']}\nTARGET MARKET: {state['target_market']}\nFUNDING STAGE: {state['funding_stage']}"
            )
        ),
    ]
    response = llm.invoke(messages).content
    pitch = parse_json_block(response, PitchBrief)
    return {
        "refined_pitch": pitch,
        "logs": ["[idea_refiner] created a structured pitch brief"],
    }


In [6]:
def competitor_analyst(state: StartupPitchState) -> dict:
    messages = [
        SystemMessage(
            content=(
                "You are a Market Intelligence Analyst. Based on the refined pitch, provide:\n"
                "1. Top 3 direct and indirect competitors in the region.\n"
                "2. Market saturation threat.\n"
                "3. Defensibility strategy.\n"
                "At the very end, include exactly two lines:\n"
                "RED_FLAG: <one sentence stating the biggest competitive threat>\n"
                "SCORE: <integer from 1 to 10 rating market opportunity>"
            )
        ),
        HumanMessage(content=f"Refined pitch:\n{state['refined_pitch']}"),
    ]
    report = llm.invoke(messages).content
    score_info = extract_score_and_flag(report, "Market")
    return {
        "competitor_analysis": report,
        "domain_scores": [score_info],
        "red_flags": [score_info["red_flag"]],
        "logs": [f"[competitor_analyst] score: {score_info['score']}/10"],
    }


In [7]:
def monetization_planner(state: StartupPitchState) -> dict:
    messages = [
        SystemMessage(
            content=(
                "You are a Startup CFO and Monetization Strategist. Based on the refined pitch, provide:\n"
                "1. 3 viable revenue models.\n"
                "2. Pricing tiers suited for the target market.\n"
                "3. Unit economics drivers and CAC vs LTV challenges.\n"
                "At the very end, include exactly two lines:\n"
                "RED_FLAG: <one sentence stating the biggest revenue or margin risk>\n"
                "SCORE: <integer from 1 to 10 rating monetization viability>"
            )
        ),
        HumanMessage(content=f"Refined pitch:\n{state['refined_pitch']}"),
    ]
    report = llm.invoke(messages).content
    score_info = extract_score_and_flag(report, "Monetization")
    return {
        "monetization_plan": report,
        "domain_scores": [score_info],
        "red_flags": [score_info["red_flag"]],
        "logs": [f"[monetization_planner] score: {score_info['score']}/10"],
    }


In [8]:
def tech_arthitect(state: StartupPitchState) -> dict:
    messages = [
        SystemMessage(
            content=(
                "You are a CTO. Based on the refined pitch, provide:\n"
                "1. Recommended technology stack.\n"
                "2. AI/LLM architecture.\n"
                "3. Cloud and API cost bottlenecks.\n"
                "At the very end, include exactly two lines:\n"
                "RED_FLAG: <one sentence stating the biggest technical or AI hallucination risk>\n"
                "SCORE: <integer from 1 to 10 rating technical feasibility>"
            )
        ),
        HumanMessage(content=f"Refined pitch:\n{state['refined_pitch']}"),
    ]
    report = llm.invoke(messages).content
    score_info = extract_score_and_flag(report, "Technology")
    return {
        "tech_blueprint": report,
        "domain_scores": [score_info],
        "red_flags": [score_info["red_flag"]],
        "logs": [f"[tech_arthitect] score: {score_info['score']}/10"],
    }


In [9]:
def regulatory_repoter(state: StartupPitchState) -> dict:
    messages = [
        SystemMessage(
            content=(
                "You are a Chief Risk & Compliance Officer. Based on the refined pitch, provide:\n"
                "1. Data privacy and compliance considerations.\n"
                "2. Supply chain or operational bottlenecks.\n"
                "3. User trust and liability hazards.\n"
                "At the very end, include exactly two lines:\n"
                "RED_FLAG: <one sentence stating the biggest legal or operational risk>\n"
                "SCORE: <integer from 1 to 10 rating regulatory and operational safety>"
            )
        ),
        HumanMessage(content=f"Refined pitch:\n{state['refined_pitch']}"),
    ]
    report = llm.invoke(messages).content
    score_info = extract_score_and_flag(report, "Regulatory")
    return {
        "regulatory_report": report,
        "domain_scores": [score_info],
        "red_flags": [score_info["red_flag"]],
        "logs": [f"[regulatory_repoter] score: {score_info['score']}/10"],
    }


In [10]:
def clamp_text(value: str, max_chars: int = 1500) -> str:
    value = (value or "").strip()
    if len(value) <= max_chars:
        return value
    trimmed = value[:max_chars]
    if "\n" in trimmed:
        trimmed = trimmed.rsplit("\n", 1)[0]
    return trimmed + "\n...[truncated]"

def vc_juge_node(state: StartupPitchState) -> dict:
    scores_list = [item["score"] for item in state["domain_scores"]]
    avg_score = round(sum(scores_list) / len(scores_list), 2) if scores_list else 0.0

    if avg_score >= 7.5:
        verdict = "INVEST (High Conviction)"
    elif avg_score >= 6.0:
        verdict = "CONDITIONAL INVEST (Requires Pivot/Mitigation)"
    else:
        verdict = "PASS (High Risk / Low Viability)"

    scorecard_str = "\n".join([f"- {item['department']}: {item['score']}/10" for item in state["domain_scores"]])
    red_flags_str = "\n".join([f"- {flag}" for flag in state["red_flags"]])
    market_report = clamp_text(state.get("competitor_analysis", ""), 900)
    monetization_report = clamp_text(state.get("monetization_plan", ""), 900)
    tech_report = clamp_text(state.get("tech_blueprint", ""), 900)
    regulatory_report = clamp_text(state.get("regulatory_report", ""), 900)

    dossier = f"""
    STARTUP IDEA: {state['raw_idea']}
    TARGET MARKET: {state['target_market']}
    FUNDING STAGE: {state['funding_stage']}

    QUANTITATIVE SCORECARD:
    {scorecard_str}
    AVERAGE SCORE: {avg_score}/10
    COMMITTEE BASELINE VERDICT: {verdict}

    CRITICAL RED FLAGS IDENTIFIED BY SPECIALISTS:
    {red_flags_str}

    SUMMARY OF SPECIALIST REPORTS:
    1. MARKET & COMPETITION:
    {market_report}

    2. MONETIZATION & UNIT ECONOMICS:
    {monetization_report}

    3. TECH & AI ARCHITECTURE:
    {tech_report}

    4. REGULATORY & OPERATIONS:
    {regulatory_report}
    """

    memo = llm.invoke([
        SystemMessage(
            content=(
                "You are the Managing Partner at a Tier-1 Venture Capital Firm. Write a crisp executive investment memo in Markdown based on the dossier. Include:\n"
                "1. Final Verdict Banner and score summary table\n"
                "2. Investment thesis and why this could win\n"
                "3. Dealbreaker risks synthesized from red flags\n"
                "4. Three mandatory milestones the founder must hit before funding."
            )
        ),
        HumanMessage(content=dossier),
    ]).content

    return {
        "final_average_score": avg_score,
        "investment_verdict": verdict,
        "vc_memo": memo,
        "pipeline_logs": [f"[Fan-In] VC Judge finalized verdict: {verdict} ({avg_score}/10)."],
    }


In [11]:
builder = StateGraph(StartupPitchState)

builder.add_node("idea_refiner", idea_refiner)
builder.add_node("competitor_analyst", competitor_analyst)
builder.add_node("monetization_planner", monetization_planner)
builder.add_node("tech_arthitect", tech_arthitect)
builder.add_node("regulatory_report", regulatory_repoter)
builder.add_node("vc_juge_node", vc_juge_node)

builder.add_edge(START, "idea_refiner")
builder.add_edge("idea_refiner", "competitor_analyst")
builder.add_edge("idea_refiner", "monetization_planner")
builder.add_edge("idea_refiner", "tech_arthitect")
builder.add_edge("idea_refiner", "regulatory_report")

builder.add_edge("competitor_analyst", "vc_juge_node")
builder.add_edge("monetization_planner", "vc_juge_node")
builder.add_edge("tech_arthitect", "vc_juge_node")
builder.add_edge("regulatory_report", "vc_juge_node")
builder.add_edge("vc_juge_node", END)

graph = builder.compile()


In [12]:
if __name__ == "__main__":
    initial_state = {
        "raw_idea": (
            "AI-powered personal chef and nutrition app in India that generates daily recipes "
            "from fridge photos, tracks macros for Indian diets, and auto-orders missing ingredients via quick-commerce apps like Blinkit and Zepto."
        ),
        "target_market": "Tier-1 and Tier-2 Urban India",
        "funding_stage": "Seed Round ($500K ask)",
        "domain_scores": [],
        "red_flags": [],
        "pipeline_logs": [],
    }

    result = graph.invoke(initial_state)

    print("=== 1. EXECUTION LOGS ===")
    for log in result.get("pipeline_logs", []):
        print(log)

    print("\n=== 2. STRUCTURED PITCH BRIEF ===")
    pitch = result["refined_pitch"]
    print(f"Problem: {pitch.problem_statement}")
    print(f"Persona: {pitch.target_persona}")
    print(f"Value Prop: {pitch.value_proposition}")

    print("\n=== 3. DEPARTMENT SCORECARD ===")
    for item in result.get("domain_scores", []):
        print(f"* {item['department']}: {item['score']}/10")

    print("\n=== 4. RED FLAGS ===")
    for flag in result.get("red_flags", []):
        print(f"! {flag}")

    print(f"\n=== 5. FINAL VERDICT: {result['investment_verdict']} (Avg: {result['final_average_score']}/10) ===")
    print(result["vc_memo"])


=== 1. EXECUTION LOGS ===
[Fan-In] VC Judge finalized verdict: PASS (High Risk / Low Viability) (5.25/10).

=== 2. STRUCTURED PITCH BRIEF ===
Problem: Urban Indian consumers struggle to plan healthy meals with the ingredients they have, leading to food waste, time‑consuming grocery trips, and poor macro‑nutrient tracking for Indian diets.
Persona: Busy, health‑conscious professionals (25‑40) living in Tier‑1 and Tier‑2 Indian cities who cook at home, own a smartphone, and regularly use quick‑commerce delivery services.
Value Prop: An AI‑powered app that instantly reads fridge photos, generates Indian‑style daily recipes, tracks macros, and auto‑orders missing ingredients via Blinkit/Zepto, saving time, reducing waste, and simplifying nutrition management.

=== 3. DEPARTMENT SCORECARD ===
* Market: 5/10
* Monetization: 5/10
* Regulatory: 6/10
* Technology: 5/10

=== 4. RED FLAGS ===
! No major red flag identified.
! No major red flag identified.
! A data‑privacy breach involving fridge 